# FleXray Demo: Render DRRs From CT And Segment X-rays

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1jMBoOyV8PkRThHi3i6QIMjolmNoRE0cD)

[FleXray](https://github.com/VictorButoi/FleXray) (`fxr`) segments anatomy in
X-ray images. It is trained on a mix of real X-rays, synthetic image+mask pairs,
and CT volumes rendered on the fly into Digitally Reconstructed Radiographs
(DRRs) -- simulated X-rays whose ground-truth labels come along for free from
the CT segmentation.

This notebook demos both halves of that story:

1. **Render a DRR from a CT volume** with `fxr.drr` -- pick a camera pose and
   ray-trace a public pelvis CT into an X-ray-like image.
2. **Segment X-rays** with the pretrained FleXray model -- load it from
   Hugging Face, predict on sample images, and view colored anatomy overlays.

**Tip:** switch to a GPU runtime (`Runtime > Change runtime type > T4 GPU`).
Everything also runs on CPU, just slower.

The setup cell installs the public FleXray release from PyPI; no GitHub login is required.

In [ ]:
%pip install -q "flexray==1.0.2" "nanodrr>=0.1.5"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.ndimage import binary_dilation, binary_erosion

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


def scale_for_display(image, lower_percentile=1.0, upper_percentile=99.0):
    """Scale a finite 2D image into ``[0, 1]`` for display.

    Args:
        image: Two-dimensional image-like array.
        lower_percentile: Percentile mapped to black.
        upper_percentile: Percentile mapped to white.

    Returns:
        Float32 NumPy array with values clipped to ``[0, 1]``.
    """
    array = np.asarray(image, dtype=np.float32)
    finite = array[np.isfinite(array)]
    if finite.size == 0:
        return np.zeros_like(array, dtype=np.float32)
    low, high = np.percentile(finite, (lower_percentile, upper_percentile))
    if high <= low:
        return np.zeros_like(array, dtype=np.float32)
    scaled = (np.nan_to_num(array, nan=low) - low) / (high - low)
    return np.clip(scaled, 0.0, 1.0)



def mask_boundary(mask, thickness=1):
    """Return an inner boundary of the requested pixel thickness.

    This follows the Figure 1 mosaic convention: erosion with a 3x3
    neighborhood and image-border pixels treated as outside the mask.

    Args:
        mask: Two-dimensional binary mask.
        thickness: Positive boundary thickness in pixels.

    Returns:
        Boolean NumPy array with True only on the mask's inner boundary.
    """
    binary = np.asarray(mask, dtype=bool)
    if binary.ndim != 2:
        raise ValueError("mask must have shape (H, W).")
    boundary_thickness = int(thickness)
    if boundary_thickness < 1:
        raise ValueError("thickness must be at least 1.")
    if not np.any(binary):
        return np.zeros_like(binary)
    neighborhood = np.ones((3, 3), dtype=bool)
    eroded = binary_erosion(
        binary,
        structure=neighborhood,
        iterations=boundary_thickness,
        border_value=0,
    )
    return binary & ~eroded


def blend_mask_layers(image, mask_weights, colors, opacity=0.5):
    """Blend every active mask into an image, including overlaps.

    Args:
        image: Grayscale or RGB image scaled to ``[0, 1]``.
        mask_weights: Nonnegative array shaped ``(C, H, W)``. Every
            nonzero channel contributes to an overlapping pixel.
        colors: Array-like sequence of ``C`` RGB colors in ``[0, 1]`` or
            ``[0, 255]``.
        opacity: Overlay opacity applied wherever at least one mask is active.

    Returns:
        RGB float32 image scaled to ``[0, 1]``. Overlapping colors are
        mixed using their mask weights instead of selecting one winner.
    """
    base = np.asarray(image, dtype=np.float32)
    if base.ndim == 2:
        base = np.repeat(base[..., None], 3, axis=-1)
    if base.ndim != 3 or base.shape[-1] != 3:
        raise ValueError("image must have shape (H, W) or (H, W, 3).")

    weights = np.clip(np.asarray(mask_weights, dtype=np.float32), 0.0, None)
    palette = np.asarray(colors, dtype=np.float32)
    if weights.ndim != 3:
        raise ValueError("mask_weights must have shape (C, H, W).")
    if palette.shape != (weights.shape[0], 3):
        raise ValueError("colors must have shape (C, 3).")
    if tuple(weights.shape[1:]) != tuple(base.shape[:2]):
        raise ValueError("mask_weights and image must have matching spatial shapes.")
    if palette.size and palette.max() > 1.0:
        palette = palette / 255.0

    total_weight = weights.sum(axis=0)
    mixed_color = np.einsum("chw,cd->hwd", weights, palette)
    np.divide(
        mixed_color,
        total_weight[..., None],
        out=mixed_color,
        where=total_weight[..., None] > 0,
    )
    alpha = float(opacity) * (total_weight > 0).astype(np.float32)
    blended = base * (1.0 - alpha[..., None]) + mixed_color * alpha[..., None]
    return np.clip(blended, 0.0, 1.0)

## Part 1: Render A DRR From A CT Volume

We download a public pelvis CT (subject 1 of the
[DeepFluoro](https://huggingface.co/datasets/eigenvivek/xvr-data) dataset,
fetched by nanoDRR) and convert TorchIO's `(W, H, D)` spatial layout to
`(D, H, W)`. The same axis permutation must be applied to the voxel-to-world
affine; permuting only the tensors rotates and misframes the DRR.

In [ ]:
import torchio as tio
from nanodrr.data.demo import download_deepfluoro

vol_path, mask_path = download_deepfluoro(subject_id=1)

img = tio.ScalarImage(vol_path)
mask_img = tio.LabelMap(mask_path)
# Reorder the data and affine axes together: TorchIO W,H,D -> fxr D,H,W.
volume_hu = img.data.permute(0, 3, 2, 1).contiguous().float().to(DEVICE)
label = mask_img.data.permute(0, 3, 2, 1)[0].contiguous().to(DEVICE)
affine = torch.as_tensor(
    img.affine[:, [2, 1, 0, 3]],
    dtype=torch.float32,
    device=DEVICE,
)

print("CT volume:", tuple(volume_hu.shape), "| label ids:", label.unique().tolist())

axial = volume_hu[0, volume_hu.shape[1] // 2].cpu()
coronal = volume_hu[0, :, volume_hu.shape[2] // 2].cpu()
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(axial, cmap="gray", origin="lower")
axes[0].set_title("axial slice")
axes[1].imshow(coronal, cmap="gray", origin="lower")
axes[1].set_title("coronal slice")
for ax in axes:
    ax.axis("off")
plt.show()

Rendering takes three ingredients: **intrinsics** (detector geometry), a
**camera pose** (here from named presets), and an **isocenter** the camera
orbits. The detector spacing and source distance below provide a wide enough
field of view for the complete pelvis while keeping cone-beam perspective.

In [ ]:
from fxr.drr import (
    DRRRenderRequest,
    PoseSampler,
    build_render_intrinsics,
    compute_isocenter,
    render_drr,
)

DRR_SDD = 1200.0
DRR_PIXEL_SPACING = 2.2
DRR_CAMERA_DISPLACEMENT = 900.0
DRR_SIZE = 256
VIEW_PRESETS = ("frontal", "lateral")

intrinsics = build_render_intrinsics(
    sdd=DRR_SDD,
    delx=DRR_PIXEL_SPACING,
    dely=DRR_PIXEL_SPACING,
    height=DRR_SIZE,
    width=DRR_SIZE,
    num_views=len(VIEW_PRESETS),
)
isocenter = compute_isocenter(volume_hu, label, affine, "label_centroid")

poses = [
    PoseSampler(preset, camera_displacement=DRR_CAMERA_DISPLACEMENT).sample()
    for preset in VIEW_PRESETS
]
rot = torch.cat([pose[0] for pose in poses], dim=0)
xyz = torch.cat([pose[1] for pose in poses], dim=0)
comparison_request = DRRRenderRequest(
    rot=rot,
    xyz=xyz,
    intrinsics=intrinsics,
    isocenter=isocenter,
    n_samples=500,
    seg_threshold=0.0,
)
comparison_result = render_drr(
    volume=volume_hu,
    label=label,
    affine=affine,
    request=comparison_request,
)

fig, axes = plt.subplots(1, len(VIEW_PRESETS), figsize=(8, 4))
for view_index, (ax, preset) in enumerate(zip(axes, VIEW_PRESETS)):
    image = comparison_result.images[view_index, 0].detach().cpu().numpy()
    ax.imshow(scale_for_display(image), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{preset} DRR")
    ax.axis("off")
plt.show()

### Your Turn: Move The Camera

Edit the variables below and re-run the cell. Presets: `frontal`, `lateral`,
`lateral_left`, `offangle`, `above`, or `random`. Detector spacing controls
the field of view; larger values show more anatomy.

In [ ]:
PRESET = "offangle"          # frontal | lateral | lateral_left | offangle | above | random
CAMERA_DISPLACEMENT = 900.0  # source distance from the isocenter (mm)
SDD = 1200.0                 # source-to-detector distance (mm)
DETECTOR_SPACING = 2.2       # detector pixel spacing (mm/px); larger = wider view
N_SAMPLES = 500              # samples per ray; lower is faster, higher is cleaner

random_params = None
if PRESET == "random":
    random_params = {
        "rot_range": {
            "alpha": [-60.0, 60.0],
            "beta": [-15.0, 15.0],
            "gamma": [-5.0, 5.0],
        },
        "xyz_range": {
            "x": [-30.0, 30.0],
            "y": [CAMERA_DISPLACEMENT, CAMERA_DISPLACEMENT],
            "z": [-30.0, 30.0],
        },
    }

rot, xyz = PoseSampler(
    PRESET,
    camera_displacement=CAMERA_DISPLACEMENT,
    sample_params=random_params,
).sample()
request = DRRRenderRequest(
    rot=rot,
    xyz=xyz,
    intrinsics=build_render_intrinsics(
        sdd=SDD,
        delx=DETECTOR_SPACING,
        dely=DETECTOR_SPACING,
        height=DRR_SIZE,
        width=DRR_SIZE,
    ),
    isocenter=isocenter,
    n_samples=N_SAMPLES,
    seg_threshold=0.0,
)
result = render_drr(
    volume=volume_hu,
    label=label,
    affine=affine,
    request=request,
)

image = result.images[0, 0].detach().cpu().numpy()
plt.figure(figsize=(4, 4))
plt.imshow(scale_for_display(image), cmap="gray", vmin=0, vmax=1)
plt.title(
    f"{PRESET} (source={CAMERA_DISPLACEMENT:.0f} mm, "
    f"sdd={SDD:.0f} mm, spacing={DETECTOR_SPACING:.1f} mm/px)"
)
plt.axis("off")
plt.show()

### Labels Come Along For Free

The same ray-tracing pass projects the CT labelmap into per-structure 2D
masks (`comparison_result.labels`, background at channel 0). A zero
projection threshold retains every ray that intersects a labeled structure.
When structures overlap in projection, every active label contributes to the
displayed color rather than one label replacing the others.

In [ ]:
frontal_index = VIEW_PRESETS.index("frontal")
drr = comparison_result.images[frontal_index, 0].detach().cpu().numpy()
projected = comparison_result.labels[frontal_index, 1:].detach().cpu().numpy()

base = scale_for_display(drr)
colors = plt.cm.tab10(np.linspace(0, 1, projected.shape[0]))[:, :3]
overlay = blend_mask_layers(base, projected, colors, opacity=0.45)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(base, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("rendered frontal DRR")
axes[1].imshow(overlay)
axes[1].set_title("projected CT labels (overlaps blended)")
for ax in axes:
    ax.axis("off")
plt.show()

## Part 2: Segment X-rays With The Pretrained Model

`FleXraySegmenter.from_pretrained` downloads the model bundle (weights, config,
label schema, preprocessing contract) from Hugging Face and returns a ready
segmenter. `predict` handles preprocessing and returns per-channel logits,
probabilities, and thresholded masks.

The bundled demo radiographs come from the permission-reviewed website
collection. Radiographs courtesy of [Radiopaedia](https://radiopaedia.org/),
used with permission.

In [ ]:
from fxr.inference import FleXraySegmenter

MODEL_ID = "VictorButoi/flexray"
segmenter = FleXraySegmenter.from_pretrained(MODEL_ID, device=DEVICE)
print(f"{len(segmenter.label_names)} output channels, e.g. {segmenter.label_names[1:6]}")

In [ ]:
from pathlib import Path
import urllib.request

ASSET_URL = (
    "https://flexray.csail.mit.edu/"
    "assets/hero-mosaic/dark/originals"
)
SAMPLE_IMAGES = {
    "chest": "slide_07.png",
    "pelvis": "slide_22.png",
}

sample_dir = Path("sample_xrays")
sample_dir.mkdir(exist_ok=True)
sample_paths = {}
for sample_name, filename in SAMPLE_IMAGES.items():
    path = sample_dir / filename
    if not path.exists():
        urllib.request.urlretrieve(f"{ASSET_URL}/{filename}", path)
    sample_paths[sample_name] = path

sample_predictions = {
    sample_name: segmenter.predict(str(path))
    for sample_name, path in sample_paths.items()
}
print("Sample X-rays:", {name: path.name for name, path in sample_paths.items()})

The next helper keeps multilabel predictions independent. Following the
Figure 1 mosaic style, the whole-label panel combines translucent filled masks
with crisp two-pixel boundaries for every above-threshold channel. Colors mix
wherever labels overlap, and the keyed legend lists only labels with visible
pixels. The selected-label panel uses the same filled-plus-contour treatment,
and each example exposes a Colab text field for choosing that label.

In [ ]:
from importlib.resources import files

from matplotlib.lines import Line2D

import yaml

_palette_yml = files("fxr.configs").joinpath(
    "colormaps", "all_structures_flexray_v4.yml"
)
_palette_raw = yaml.safe_load(_palette_yml.read_text(encoding="utf-8"))["labels"]
PALETTE = {
    name: None
    if color is None
    else tuple(int(color[index : index + 2], 16) for index in (1, 3, 5))
    for name, color in _palette_raw.items()
}


def prepared_image_for_display(image_path, preprocessing):
    """Return the exact spatially prepared model input for display.

    Args:
        image_path: Path to an input radiograph.
        preprocessing: `ImagePreprocessing` contract used by the segmenter.

    Returns:
        Grayscale float32 array shaped like the prediction and scaled to
        ``[0, 1]``.
    """
    prepared = preprocessing.prepare(image_path)[0, 0].detach().cpu().numpy()
    if preprocessing.std is not None:
        prepared = prepared * float(preprocessing.std)
    if preprocessing.mean is not None:
        prepared = prepared + float(preprocessing.mean)
    if preprocessing.scale == "none":
        prepared = prepared / 255.0
    return np.clip(prepared, 0.0, 1.0).astype(np.float32)


def prediction_filled_contour_overlay(
    base,
    prediction,
    label_names,
    selected_labels=None,
    threshold=0.5,
    fill_opacity=0.58,
    boundary_opacity=0.95,
    boundary_pixels=2,
    boundary_outline_opacity=0.85,
    boundary_outline_pixels=0,
):
    """Create a filled multilabel overlay with crisp prediction contours.

    Translucent fills are drawn first and a near-opaque two-pixel boundary is
    drawn last. An optional dilated outline can be enabled by setting
    boundary_outline_pixels above zero. Every layer mixes coincident labels
    instead of selecting one winner.

    Args:
        base: Prepared grayscale image scaled to [0, 1].
        prediction: FleXrayPrediction containing per-channel probabilities.
        label_names: Ordered channel names, including background at index 0.
        selected_labels: Optional label-name sequence. None selects every
            foreground label.
        threshold: Minimum channel probability included in the overlay.
        fill_opacity: Opacity of the filled mask layer.
        boundary_opacity: Opacity of the boundary layer.
        boundary_pixels: Positive inner-boundary thickness in pixels.
        boundary_outline_opacity: Opacity of the optional dilated outline.
        boundary_outline_pixels: Dilation radius for the optional outline.
            The zero default keeps contours thin.

    Returns:
        Pair of the RGB overlay and the tuple of selected labels with at least
        one above-threshold pixel.
    """
    opacities = {
        "fill_opacity": fill_opacity,
        "boundary_opacity": boundary_opacity,
        "boundary_outline_opacity": boundary_outline_opacity,
    }
    for opacity_name, opacity_value in opacities.items():
        if not 0.0 <= float(opacity_value) <= 1.0:
            raise ValueError(f"{opacity_name} must be in [0, 1].")
    boundary_thickness = int(boundary_pixels)
    if boundary_thickness < 1:
        raise ValueError("boundary_pixels must be at least 1.")
    outline_pixels = int(boundary_outline_pixels)
    if outline_pixels < 0:
        raise ValueError("boundary_outline_pixels must be nonnegative.")

    names = tuple(label_names)
    chosen = tuple(names[1:] if selected_labels is None else selected_labels)
    unknown = sorted(set(chosen) - set(names[1:]))
    if unknown:
        raise ValueError(
            f"Unknown label(s) {unknown}. Choose from: {', '.join(names[1:])}"
        )

    indices = [names.index(name) for name in chosen]
    probabilities = prediction.probabilities[0, indices].detach().cpu().numpy()
    weights = np.where(probabilities >= threshold, probabilities, 0.0)
    masks = weights > 0.0
    colors = [PALETTE[name] for name in chosen]

    overlay = blend_mask_layers(
        base,
        weights,
        colors,
        opacity=fill_opacity,
    )
    boundaries = np.stack(
        [
            mask_boundary(mask, thickness=boundary_thickness)
            for mask in masks
        ],
        axis=0,
    )
    if outline_pixels > 0:
        neighborhood = np.ones((3, 3), dtype=bool)
        outlines = np.stack(
            [
                binary_dilation(
                    boundary,
                    structure=neighborhood,
                    iterations=outline_pixels,
                )
                for boundary in boundaries
            ],
            axis=0,
        )
        overlay = blend_mask_layers(
            overlay,
            outlines.astype(np.float32),
            colors,
            opacity=boundary_outline_opacity,
        )
    overlay = blend_mask_layers(
        overlay,
        boundaries.astype(np.float32),
        colors,
        opacity=boundary_opacity,
    )

    visible = tuple(name for name, mask in zip(chosen, masks) if np.any(mask))
    return overlay, visible


def show_prediction_triptych(
    image_path,
    prediction,
    segmenter,
    individual_label,
    threshold=0.5,
    opacity=0.5,
    title=None,
):
    """Show the input, all predictions, and one selected filled-and-contoured label.

    Args:
        image_path: Path to the source radiograph.
        prediction: Prediction returned by segmenter.predict.
        segmenter: Loaded FleXraySegmenter with labels and preprocessing.
        individual_label: Foreground label to show in the third panel.
        threshold: Minimum probability included in either overlay.
        opacity: Filled-overlay opacity for the selected-label panel.
        title: Optional descriptive title for the input panel.

    Returns:
        Tuple of foreground labels visible in the all-label panel and legend.
    """
    base = prepared_image_for_display(image_path, segmenter.preprocessing)
    whole_overlay, visible = prediction_filled_contour_overlay(
        base,
        prediction,
        segmenter.label_names,
        threshold=threshold,
    )
    single_overlay, selected_visible = prediction_filled_contour_overlay(
        base,
        prediction,
        segmenter.label_names,
        selected_labels=(individual_label,),
        threshold=threshold,
        fill_opacity=opacity,
    )

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(base, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title(title or Path(image_path).name)
    axes[1].imshow(whole_overlay, interpolation="nearest")
    axes[1].set_title("all labels (filled + 2 px contours)")
    axes[2].imshow(single_overlay, interpolation="nearest")
    status = "above threshold" if selected_visible else "no pixels above threshold"
    axes[2].set_title(f"{individual_label} ({status})")
    for ax in axes:
        ax.axis("off")

    if visible:
        handles = [
            Line2D(
                [],
                [],
                color=np.asarray(PALETTE[name], dtype=np.float32) / 255.0,
                linewidth=2.0,
                label=name,
            )
            for name in visible
        ]
        legend_columns = min(6, len(handles))
        legend_rows = (len(handles) + legend_columns - 1) // legend_columns
        width, height = fig.get_size_inches()
        legend_height = 0.35 + 0.24 * legend_rows
        fig.set_size_inches(width, height + legend_height, forward=True)
        fig.legend(
            handles=handles,
            loc="lower center",
            bbox_to_anchor=(0.5, 0.01),
            ncol=legend_columns,
            title="Predicted labels",
            frameon=False,
            fontsize="small",
            title_fontsize="small",
            handlelength=1.5,
            columnspacing=1.2,
            borderaxespad=0.0,
        )
        fig.tight_layout(rect=(0, legend_height / (height + legend_height), 1, 1))
    else:
        fig.tight_layout()

    plt.show()
    return visible

### Lateral Chest X-ray

Change `CHEST_LABEL` and re-run this cell to inspect another structure
without recomputing the prediction.

In [ ]:
CHEST_LABEL = "lungs"  # @param {type:"string"}

chest_visible = show_prediction_triptych(
    sample_paths["chest"],
    sample_predictions["chest"],
    segmenter,
    individual_label=CHEST_LABEL,
    title="Lateral chest X-ray",
)
print("Labels above threshold:", ", ".join(chest_visible) or "none")

### Pelvis X-ray

Change `PELVIS_LABEL` and re-run this cell to inspect another structure
without recomputing the prediction.

In [ ]:
PELVIS_LABEL = "hips"  # @param {type:"string"}

pelvis_visible = show_prediction_triptych(
    sample_paths["pelvis"],
    sample_predictions["pelvis"],
    segmenter,
    individual_label=PELVIS_LABEL,
    title="Pelvis X-ray",
)
print("Labels above threshold:", ", ".join(pelvis_visible) or "none")

### Your Turn: Segment Your Own X-ray

Choose a label in the text field, run the cell, and upload any PNG/JPEG
radiograph. The same input / whole overlay / individual-label view is used.

In [ ]:
UPLOAD_LABEL = "lungs"  # @param {type:"string"}

try:
    from google.colab import files as colab_files
except ImportError:
    print("Not running in Colab -- call segmenter.predict(<path>) locally instead.")
else:
    for name in colab_files.upload():
        prediction = segmenter.predict(name)
        visible = show_prediction_triptych(
            name,
            prediction,
            segmenter,
            individual_label=UPLOAD_LABEL,
            title=name,
        )
        print("Labels above threshold:", ", ".join(visible) or "none")

## Where To Go Next

- **Package + docs:** [github.com/VictorButoi/FleXray](https://github.com/VictorButoi/FleXray)
  -- see `docs/drr.md` for the rendering primitives and `docs/inference.md` for
  the prediction API and `flexify` CLI.
- **Train your own model:** `docs/training.md` in the FleXray repository
  covers mixing real X-rays, online CT->DRR rendering, and generated data.
- Model weights are released under `CC-BY-NC-4.0`; code is MIT.